Exercise 3: Frame-Autoregressive Pong



[← Back to reading list]()

# Exercise 3: Frame-Autoregressive Pong

> **Learning objectives:**

**Learning objectives:**


- Extend the image DiT to handle video (sequences of frames)

- Implement block-causal attention for frame-autoregressive generation

- Add action conditioning for controllable world models

- Understand and implement diffusion forcing for video training

- Generate Pong video sequences from a trained model

**Prerequisites:** [Exercise 2]() (working MNIST flow matching model). Reference implementation: [wendlerc/toy-wm]().

## Setup

In [ ]:
import torch
import torch as t
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from tqdm import tqdm
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

## From Images to Video: What Changes?

Going from MNIST images to Pong videos, almost everything stays the same—we just add a **temporal dimension**. Here's the mapping:

ComponentImage (Ex. 2)Video (Ex. 3)
  Input shape`(B, C, H, W)``(B, T, C, H, W)`
  Patch tokens`(B, h*w, d)``(B, T*h*w, d)`
  AttentionBidirectional (all-to-all)Block-causal (frame \(t\) attends to frames \(\le t\))
  ConditioningClass labelAction per frame
  TrainingOne noise level per imageDifferent noise levels per frame (diffusion forcing)

## Part A: Dataset

---

### Exercise 1 — Load and Visualize the Pong Dataset

Difficulty: 🔴🔴⭕⭕⭕  |  Importance: 🔵🔵🔵⭕⭕  |  ~15 min

Download the Pong dataset from HuggingFace and create a DataLoader. The dataset contains sequences of frames with corresponding actions.

In [ ]:
from huggingface_hub import hf_hub_download
import os

# Download dataset
os.makedirs("datasets/pong1M", exist_ok=True)
for fname in ["frames.npy", "actions.npy"]:
    hf_hub_download(repo_id="wendlerc/pong1M", filename=fname,
                    repo_type="dataset", local_dir="datasets/pong1M")

# Load raw data
all_frames = np.load("datasets/pong1M/frames.npy")   # (N, C, H, W) uint8
all_actions = np.load("datasets/pong1M/actions.npy")  # (N,) int
print(f"Frames: {all_frames.shape}, Actions: {all_actions.shape}")
print(f"Action values: {np.unique(all_actions)}")

Now create a Dataset that returns sequences of consecutive frames:

In [ ]:
class PongDataset(Dataset):
    """Dataset of Pong video sequences.

    Each item is a sequence of `seq_len` consecutive frames and their actions.
    Actions: 0=dummy (for CFG), 1=noop, 2=up, 3=down
    """
    def __init__(self, frames, actions, seq_len=17, fps=5):
        super().__init__()
        # YOUR CODE HERE
        # 1. Reshape frames into episodes of (fps*duration + 1) consecutive frames
        #    (the +1 is because we need seq_len frames starting from any position)
        # 2. Normalize frames to [-1, 1]
        # 3. Store sequences of length seq_len

    def __len__(self):
        # YOUR CODE HERE
        pass

    def __getitem__(self, idx):
        # YOUR CODE HERE
        # Return (frames, actions) where:
        #   frames: (seq_len, C, H, W) float tensor in [-1, 1]
        #   actions: (seq_len,) long tensor
        pass

Visualize a sample sequence:

In [ ]:
# YOUR CODE HERE
# Display the first 8 frames of a sequence as a row of images
# Add the action label above each frame

## Part B: Video Patch Embedding

---

### Exercise 2 — Extend Patch/UnPatch for Video

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~20 min

Modify the Patch and UnPatch modules to handle 5D video tensors `(B, T, C, H, W)`. The simplest approach: reshape `(B, T, C, H, W)` to `(B*T, C, H, W)`, apply the image patchifier, then reshape back to `(B, T*n_patches, d)`.

In [ ]:
class VideoPatch(nn.Module):
    """Patchify video frames into a single token sequence."""
    def __init__(self, patch_size=4, in_channels=3, d=64):
        super().__init__()
        self.patch_size = patch_size
        self.d = d
        # YOUR CODE HERE
        # Same conv as image Patch

    def forward(self, x):
        """
        Args:
            x: (B, T, C, H, W) video tensor
        Returns:
            (B, T * n_patches_per_frame, d) token sequence
        """
        # YOUR CODE HERE
        pass

    def n_patches_per_frame(self, h, w):
        return (h // self.patch_size) * (w // self.patch_size)


class VideoUnPatch(nn.Module):
    """Convert token sequence back to video frames."""
    def __init__(self, patch_size=4, d=64, out_channels=3):
        super().__init__()
        self.patch_size = patch_size
        self.out_channels = out_channels
        # YOUR CODE HERE

    def forward(self, x, n_frames):
        """
        Args:
            x: (B, T * n_patches_per_frame, d) token sequence
            n_frames: number of frames T
        Returns:
            (B, T, C, H, W) video tensor
        """
        # YOUR CODE HERE
        pass

Test:

In [ ]:
vpatch = VideoPatch(patch_size=4, in_channels=3, d=64)
vunpatch = VideoUnPatch(patch_size=4, d=64, out_channels=3)

video = t.randn(2, 5, 3, 24, 24)  # 2 videos, 5 frames, 3 channels, 24x24
tokens = vpatch(video)
n_patches = vpatch.n_patches_per_frame(24, 24)
assert tokens.shape == (2, 5 * n_patches, 64), f"Got {tokens.shape}"
reconstructed = vunpatch(tokens, n_frames=5)
assert reconstructed.shape == (2, 5, 3, 24, 24), f"Got {reconstructed.shape}"
print(f"Video {video.shape} -> Tokens {tokens.shape} -> Video {reconstructed.shape}")

## Part C: Block-Causal Attention

In the image model, every patch attends to every other patch. For video, we need **block-causal attention**: patches in frame \(t\) can attend to all patches in frames \(\le t\), but not to future frames. This enables autoregressive generation frame-by-frame.

---

### Exercise 3 — Implement Block-Causal Attention Mask

Difficulty: 🔴🔴🔴🔴⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~20 min

Create a function that generates a block-causal mask. If each frame has \(P\) patches and there are \(T\) frames, the total sequence length is \(S = T \times P\). Token \(i\) belongs to frame \(\lfloor i/P \rfloor\). Token \(i\) can attend to token \(j\) if and only if \(\lfloor j/P \rfloor \le \lfloor i/P \rfloor\).

In [ ]:
def make_block_causal_mask(n_frames, patches_per_frame, device="cpu"):
    """Create a block-causal attention mask.

    Args:
        n_frames: number of frames T
        patches_per_frame: number of patches per frame P
        device: torch device

    Returns:
        Boolean mask of shape (T*P, T*P) where True means BLOCKED (masked out)
    """
    # YOUR CODE HERE
    pass

Test and visualize:

In [ ]:
mask = make_block_causal_mask(4, 9)  # 4 frames, 9 patches each
plt.figure(figsize=(6, 6))
plt.imshow(~mask.cpu(), cmap='gray')
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Block-causal mask (white = attend)")
# Add grid lines at frame boundaries
for i in range(1, 4):
    plt.axhline(y=i*9-0.5, color='red', linewidth=0.5)
    plt.axvline(x=i*9-0.5, color='red', linewidth=0.5)
plt.show()

---

### Exercise 4 — Implement Causal Video Attention

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

Modify the Attention module to accept an optional mask parameter for block-causal masking.

In [ ]:
class CausalVideoAttention(nn.Module):
    def __init__(self, d=64, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d = d
        self.d_head = d // n_head
        self.QKV = nn.Linear(d, 3 * d, bias=False)
        self.O = nn.Linear(d, d, bias=False)
        self.normq = RMSNorm(self.d_head)
        self.normk = RMSNorm(self.d_head)

    def forward(self, x, mask=None):
        """
        Args:
            x: (B, S, d) where S = T * patches_per_frame
            mask: (S, S) bool tensor, True = blocked
        Returns:
            (B, S, d)
        """
        # YOUR CODE HERE
        # Same as the image Attention but apply the mask before softmax
        pass

## Part D: The Causal DiT for Video

---

### Exercise 5 — Build the Causal Video DiT

Difficulty: 🔴🔴🔴🔴⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~30 min

Assemble the full video DiT. Key differences from the image version:


- **Input:** `(B, T, C, H, W)` video with per-frame timesteps and actions

- **Conditioning:** each frame gets its own timestep and action embedding. The conditioning is applied *per-token* (each patch inherits its frame's conditioning)

- **Positional embedding:** spatial positional embedding (same for each frame) tiled across frames

- **Attention mask:** block-causal

In [ ]:
class CausalDiT(nn.Module):
    def __init__(self, h=24, w=24, n_actions=4, in_channels=3,
                 patch_size=4, n_blocks=8, d=64, n_head=4, exp=2, T=1000):
        super().__init__()
        self.T = T
        self.patches_per_frame = (h // patch_size) * (w // patch_size)
        # YOUR CODE HERE
        # - VideoPatch and VideoUnPatch
        # - Spatial positional embedding: (1, patches_per_frame, d)
        # - NumEmbedding for timesteps
        # - nn.Embedding for actions (n_actions entries)
        # - SiLU activation
        # - DiT blocks (use CausalVideoAttention in the blocks)
        # - Final norm + modulation + unpatch

    def forward(self, x, actions, ts):
        """
        Args:
            x: (B, T_frames, C, H, W) noisy video
            actions: (B, T_frames) action indices per frame
            ts: (B, T_frames) float timesteps per frame (each frame can have
                a different noise level - this is diffusion forcing!)
        Returns:
            (B, T_frames, C, H, W) predicted velocity per frame
        """
        # YOUR CODE HERE
        # 1. Patchify: (B, T*P, d)
        # 2. Add positional embedding (tile spatial PE across frames)
        # 3. Build per-token conditioning:
        #    - Compute per-frame cond = SiLU(timestep_emb + action_emb)  -> (B, T, d)
        #    - Repeat for each patch: (B, T, d) -> (B, T*P, d)
        # 4. Build block-causal mask
        # 5. Run through DiT blocks (pass mask to attention)
        # 6. Final modulated norm + unpatch
        pass

## Part E: Diffusion Forcing Training

In standard flow matching (Exercise 2), each image gets one noise level \(t\). For **diffusion forcing**, each frame in the video gets an *independent* noise level. This is crucial because during autoregressive generation, earlier frames will be cleaner than later ones.

---

### Exercise 6 — Implement the Diffusion Forcing Loss

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~15 min

Like flow matching, but with per-frame timesteps. Also apply action dropout (20%) for CFG.

In [ ]:
def diffusion_forcing_loss(model, frames, actions, action_dropout=0.2):
    """Compute diffusion forcing loss for video.

    Args:
        model: CausalDiT
        frames: (B, T, C, H, W) clean video frames
        actions: (B, T) action indices
        action_dropout: probability of dropping action (setting to 0)

    Returns:
        scalar loss
    """
    # YOUR CODE HERE
    # 1. Sample per-frame timesteps: ts of shape (B, T) ~ U(0, 1)
    # 2. Sample noise z ~ N(0, I) same shape as frames
    # 3. Compute per-frame noisy frames: x_t = frames - ts * (frames - z)
    #    (need to broadcast ts to match spatial dims)
    # 4. Apply action dropout
    # 5. Predict velocity, compute MSE loss
    pass

---

### Exercise 7 — Train the Pong Model

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵⭕  |  ~15 min (+ training time)

Write the training loop. This uses a similar structure to MNIST but with the diffusion forcing loss.

In [ ]:
model = CausalDiT(h=24, w=24, n_actions=4, in_channels=3,
                   patch_size=4, n_blocks=8, d=64, n_head=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95))

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# YOUR CODE HERE
# Training loop:
# - Train for ~2500 steps (or more for better quality)
# - For each batch: compute diffusion_forcing_loss, backprop, clip grads, step
# - Log loss every 100 steps
# - Generate sample videos every 500 steps

---

### Exercise 8 — Implement Video Sampling

Difficulty: 🔴🔴🔴🔴⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~25 min

Video sampling is done **frame-by-frame** (autoregressively). For each new frame:


- Start with noise for the new frame

- Concatenate with all previously generated (clean) frames

- Run denoising steps: the model sees all frames, but only the last frame's noise level changes from 1→0

- After denoising, the new frame is "clean" and becomes context for the next frame

In [ ]:
@t.no_grad()
def sample_video(model, first_frame, actions, n_denoise_steps=30, cfg=0):
    """Generate video autoregressively, one frame at a time.

    Args:
        model: trained CausalDiT
        first_frame: (B, 1, C, H, W) the initial context frame
        actions: (B, total_frames) action sequence (including first frame)
        n_denoise_steps: Euler steps per frame
        cfg: classifier-free guidance strength

    Returns:
        (B, total_frames, C, H, W) generated video
    """
    B = first_frame.shape[0]
    total_frames = actions.shape[1]
    C, H, W = first_frame.shape[2:]

    # Start with the given first frame
    video = first_frame.clone()  # (B, 1, C, H, W)

    for frame_idx in range(1, total_frames):
        # YOUR CODE HERE
        # 1. Sample noise for the new frame: z ~ N(0, I)
        # 2. Set up timesteps: all previous frames at t=0, new frame starts at t=1
        # 3. Run n_denoise_steps Euler steps:
        #    a. Concatenate clean frames + current noisy frame
        #    b. Set ts for all frames (0 for clean, current t for new)
        #    c. Predict velocity (apply CFG if needed)
        #    d. Only update the new frame: z += (t_i - t_{i+1}) * v_pred[:, -1:]
        # 4. Append denoised frame to video
        pass

    return video

---

### Exercise 9 — Visualize Generated Pong Videos

Difficulty: 🔴⭕⭕⭕⭕  |  Importance: 🔵🔵🔵⭕⭕  |  ~10 min

Generate and visualize Pong videos. Use the first frame from the dataset as context.

In [ ]:
# YOUR CODE HERE
# 1. Get a sample from the dataset
# 2. Use the first frame as context
# 3. Generate a video with sample_video()
# 4. Display as an animation or a grid of frames

# Helper to create an animation:
def show_video(frames, fps=5):
    """Display a video tensor (T, C, H, W) as an animation."""
    frames_np = ((frames.cpu().clamp(-1, 1) + 1) / 2).permute(0, 2, 3, 1).numpy()
    fig, ax = plt.subplots()
    im = ax.imshow(frames_np[0].clip(0, 1))
    ax.axis('off')
    def update(i):
        im.set_data(frames_np[i].clip(0, 1))
        return [im]
    ani = animation.FuncAnimation(fig, update, frames=len(frames_np), interval=1000//fps)
    plt.close()
    return HTML(ani.to_jshtml())

## Summary

You've built a frame-autoregressive video model for Pong! The key extensions from the image model were:


- **Temporal dimension:** Video tensors `(B, T, C, H, W)` flattened into longer token sequences

- **Block-causal attention:** Each frame attends to itself and all previous frames

- **Per-frame conditioning:** Action and timestep embeddings applied per-frame

- **Diffusion forcing:** Independent noise levels per frame during training

- **Autoregressive sampling:** Generating one frame at a time, using clean past frames as context

In [Exercise 4](), we'll add KV-caching to make this inference process much faster.

[← Previous: Exercise 2]()   [Next: Exercise 4 →]()